# 1. Agentes con Memoria Conversacional en LangChain

## Objetivos de Aprendizaje
- Comprender por qué la memoria es crucial para crear agentes conversacionales efectivos.
- Aprender a gestionar el historial de una conversación (`chat_history`) con el `AgentExecutor`.
- Implementar un agente que recuerde interacciones pasadas para responder preguntas de seguimiento.
- Entender cómo LangChain pasa el contexto de la conversación al LLM.

## ¿Qué es la Memoria y Por Qué es Importante?

Por defecto, los LLMs y los agentes que hemos construido hasta ahora **no tienen estado (stateless)**. Cada vez que los invocamos, procesan la solicitud como si fuera la primera vez que interactúan con nosotros. No tienen recuerdo de preguntas o respuestas anteriores.

Esto es una gran limitación para crear asistentes o chatbots útiles. Un usuario espera poder hacer preguntas de seguimiento, referirse a información mencionada previamente y tener una conversación fluida. 

La **memoria** es el mecanismo que permite a un agente recordar interacciones pasadas. LangChain facilita enormemente la gestión de esta memoria. La forma más común de memoria es el **historial de chat (chat history)**, donde simplemente guardamos la lista de todos los mensajes de la conversación.

En este notebook, veremos cómo añadir esta capacidad a nuestro agente de LangChain.

### 1. Instalación y Configuración

In [1]:
# Instalación de dependencias.
#
# Solo hace falta en Google Colab. En local, `uv sync` ya instaló todo esto con las
# versiones exactas del uv.lock; lanzar pip con -U aquí las actualizaría y rompería
# la reproducibilidad que el curso garantiza a todo el grupo.
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain-groq groq langgraph langchain langchain-classic wikipedia python-dotenv
else:
    print("Entorno local: las dependencias ya las instaló uv sync.")


Entorno local: las dependencias ya las instaló uv sync.


In [2]:
import os
import wikipedia
from langchain_groq import ChatGroq

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# Agente de LangChain con herramientas nativas (create_openai_tools_agent).
# Por esta vía el modelo grande es el fiable: medido sobre 6 consultas, llama-3.3-70b
# acertó 6/6 el formato de la llamada a la función y llama-3.1-8b solo 4/6.
# (Ojo: con el SDK crudo de Groq la relación se invierte; ver 2-agent-function-calling.)
MODELO = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM (ChatGroq lee GROQ_API_KEY del entorno)
try:
    llm = ChatGroq(
        model=MODELO,
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Herramientas y Agente (Sin Cambios)

La definición de las herramientas y la creación del agente son exactamente las mismas que en el notebook anterior. La magia de la memoria no está en la definición del agente, sino en **cómo lo ejecutamos**.

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
# El prompt del agente, definido aquí en vez de descargarlo del hub de LangChain.
#
# Antes esto era `hub.pull("hwchase17/openai-tools-agent")`. Se cambió por tres razones:
#   1. Seguridad: descargar un prompt público deserializa objetos de LangChain de un
#      tercero. LangChain lo bloqueó por defecto justo por eso.
#   2. Fiabilidad: `hub.pull` necesita red; sin internet el notebook no arranca.
#   3. Didáctica: así ves el prompt real que gobierna al agente, que es lo interesante.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


### 3. Gestionando la Memoria Conversacional

Para que el agente recuerde, necesitamos hacer dos cosas:

1.  **Mantener un historial**: Crearemos una lista llamada `chat_history` para almacenar los mensajes.
2.  **Pasar el historial en cada llamada**: El `AgentExecutor` acepta un parámetro `chat_history`. LangChain se encarga de formatear esta lista y añadirla al prompt que se envía al LLM.

El formato del historial es una lista de objetos `BaseMessage` de LangChain. Los más comunes son `HumanMessage` (para el usuario) y `AIMessage` (para la respuesta del agente).

In [4]:
from langchain_core.messages import HumanMessage, AIMessage

# Iniciamos el historial de chat como una lista vacía
chat_history = []

#### Primera Interacción: Sin Historial

In [5]:
query1 = "Háblame del planeta Saturno"

response1 = agent_executor.invoke({
    "input": query1,
    "chat_history": chat_history
})

print(f"Respuesta 1: {response1['output']}")



> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Saturno'}`




/Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Ocurrió un error: "Saturno" may refer to: 
Saturno (mitología)
Saturno (planeta)
Saturno (Buenos Aires)
Saturno (Rubens)
Saturno (cohete)
Saturno I
Saturno IB
Saturno V
Sombrero saturno
Sergio Saturno
Saturno (álbum)
Wikcionario


Invoking: `get_wikipedia_summary` with `{'query': 'Saturno planeta'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Saturn'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Saturno es un planeta del sistema solar conocido por sus anillos. Es el sexto planeta más cercano al Sol y el segundo más grande del sistema solar, después de Júpiter. Saturno tiene al menos 62 lunas y su atmósfera está compuesta principalmente de hidrógeno y helio. El planeta es conocido por sus fuertes vientos y tormentas, y su temperatura promedio es de alrededor de -178°C. Saturno es un planeta gaseoso, lo que significa que no tiene una superficie sólida y está compuesto principalmente de gases. El planeta es estudiado por astrónomos y científicos para aprender más sobre su formación, evolución y características únicas.

> Finished chain.
Respuesta 1: Saturno es un planeta del sistema solar conocido por sus anillos. Es el sexto planeta más cercano al Sol y el segundo más grande del sistema solar, después de Júpiter. Saturno tiene al menos 62 lunas y su atmósfera está compuesta principalmente de hidrógeno y helio. El planeta es conocido por sus fuertes vientos y tormentas, y su temp

Ahora, actualizamos manualmente nuestro historial con la pregunta del usuario y la respuesta del agente.

In [6]:
chat_history.append(HumanMessage(content=query1))
chat_history.append(AIMessage(content=response1["output"]))

print("Historial actualizado.")

Historial actualizado.


#### Segunda Interacción: Con Historial

Ahora hacemos una pregunta de seguimiento. Fíjate que no mencionamos "Saturno", simplemente preguntamos "¿de qué están hechos sus anillos?"

In [7]:
query2 = "¿Y de qué están hechos sus anillos?"

response2 = agent_executor.invoke({
    "input": query2,
    "chat_history": chat_history
})

print(f"Respuesta 2: {response2['output']}")



> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'anillos de Saturno composicion'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'composición de los anillos de Saturno'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'anillos de Saturno'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Los anillos de Saturno están compuestos principalmente por partículas de hielo y roca. Estas partículas varían en tamaño, desde pequeños granos de polvo hasta bloques de hielo de varios metros de diámetro. Los anillos también contienen una pequeña cantidad de polvo y partículas más finas, que son el resultado de colisiones entre las partículas más grandes. La composición exacta de los anillos de Saturno es objeto de estudio y debate entre los científicos, pero en general se cree que están formados por una mezcla de hielo de agua, hielo de amoníaco y partículas de roca.

> Finished chain.
Respuesta 2: Los anillos de Saturno están compuestos principalmente por partículas de hielo y roca. Estas partículas varían en tamaño, desde pequeños granos de polvo hasta bloques de hielo de varios metros de diámetro. Los anillos también contienen una pequeña cantidad de polvo y partículas más finas, que son el resultado de colisiones entre las partículas más grandes. La composición exacta de los anil

¡Funcionó! El agente entendió que "sus anillos" se refería a los anillos de Saturno, porque la conversación anterior estaba en su contexto. El `verbose=True` nos muestra que el agente decidió buscar en Wikipedia "anillos de Saturno", combinando la nueva pregunta con el historial.

## Conclusiones

Añadir memoria a un agente de LangChain es sorprendentemente sencillo, pero increíblemente poderoso. Simplemente manteniendo una lista del historial de chat y pasándola en cada invocación, transformamos un agente de una sola respuesta en un verdadero **asistente conversacional**.

La clave es que los componentes de LangChain (`AgentExecutor`, los prompts de `hub`) ya están diseñados para buscar y utilizar la variable `chat_history` si se proporciona.

**Limitaciones:**
- **Gestión Manual**: En este ejemplo, actualizamos la lista `chat_history` manualmente. Para una aplicación real, querríamos encapsular esto en una clase o función.
- **Tamaño del Contexto**: Enviar el historial completo en cada llamada puede volverse costoso y exceder el límite de tokens del modelo en conversaciones muy largas.

En los próximos notebooks, exploraremos los **sistemas de memoria** que LangChain ofrece para gestionar estas limitaciones, como la memoria de búfer (`BufferMemory`) que automatiza la gestión del historial y la memoria de resumen (`SummaryMemory`) que condensa conversaciones largas para ahorrar tokens.